# Imports

In [167]:
import numpy as np
import itertools
from numpy import random as random

In [168]:
rng = random.default_rng()

### Let's start with constant h and see what happens

In [ ]:
def pt(sigmat,use_X,**X):
    '''
    Input:
        - sigmat: a (N,) vector of the response of N neurons at a particular time t, where sigmat[i] corresponds to the response of the ith neuron
        - X: a (B*N + N*(N-1)/2 , ) vector containing:
            - h: (B*N,) vector of the time-dependent field, where h[N*t + n] corresponds to the time dependent field of the nth neuron at time t
            - J: (N(N-1)/2,) vector of the fixed couplings between two neurons

    Output:
        pt: the probability of that state at that time, given the parameters
    '''
    
    N = sigmat.shape[0]
    NN = int(N*(N-1)/2)

    if use_X == True:
        X_m = X['X']
        J = X_m[-NN:]
        h = X_m[:-NN]
        h = h.reshape((-1,N))
        ht = h[X['t']]
    else:
        ht = X['ht']
        J = X['J']
    Jtmat = np.zeros((N,N))
    Jt_indices = np.tril_indices(N-1)
    Jtmat[Jt_indices[0]+1,Jt_indices[1]] = J

    corr_mat = np.outer(sigmat,sigmat)
    corr_mat[np.triu_indices(N)] = 0

    corr_mat = corr_mat*Jtmat

    sum_couplings = sum((Jtmat*corr_mat)@sigmat)

    sum_fields = ht@sigmat

    pt = np.exp(-1*(sum_fields+sum_couplings))
    
    return pt

def observables(sigmab):
    '''
    Input:
        - sigmab: an (N,) vector of binarized neural responses at time b; sigmab[n] is the response of neuron n
    
    Output:
        - observables: (N + N*(N-1)/2 , ) vector of the observables at that point in time
    '''
    corr = np.outer(sigmab,sigmab)
    corr_vec = corr[np.tril_indices(corr.shape[0],-1)]
    return np.concatenate((sigmab,corr_vec))

def P_bar(sigma):
    ''' 
    Input:
        sigma: a (B,N) matrix of binarized neural responses, where sigma[b,n] is the response of neuron n at time b

    Output:
        P_b: a (N + N*(N-1)/2 , ) vector of the average of the observables given the neural data
    '''

    P_b = np.array(list(map(observables,sigma))).mean(axis=0)

    return P_b

def QX(X):
    '''
    Input:
        - X: a (B*N + N*(N-1)/2 , ) vector of the parameters of the model
    
    Output:
        Q: a (N + N*(N-1)/2 , ) vector of the model averages of the observables.
    '''

    N = 20
    NN = int(N*(N-1)/2)
    h = X[:-NN]
    J = X[-NN:]
    h = h.reshape((-1,N))
    h_bar = h.mean(axis=0)

    combs = np.fromiter(itertools.product(range(2),repeat = N),dtype=np.dtype((np.float32,N)),count=2**N)

    weighted_observables = map(lambda x : observables(x)*pt(x,use_X = False,ht = h_bar,J=J), combs)

    wo_np = np.fromiter(weighted_observables,dtype=np.dtype((np.float32,int(N + N*(N-1)/2))),count=2**N)

    Q = wo_np.sum(axis=0)

    return Q

def QMC(sigma,M):
    '''
    Input:
        - sigma: a (B,N) matrix of binarized neural responses, where sigma[b,n] is the response of neuron n at time b
        - M: the number of times to sample from sigma
    
    Output:
        QMC: a montecarlo approximation of Q
    '''
    MC = rng.choice(sigma,M,replace=True)
    QMC = map(observables,MC)
    QMC = np.fromiter(QMC,dtype = np.dtype((np.float32,int(N + N*(N-1)/2))),count=MC)
    QMC = QMC.mean(axis=0)

    return QMC



In [161]:
B = 50
N = 20
t = 2
sigma = np.random.rand(B,N)
X = np.random.rand(int(B*N + N*(N-1)/2))